# UniProt EDA

Raw parquet from `python src/ingest_uniprot.py`.


In [1]:
from pathlib import Path
import pandas as pd
import os
from dotenv import load_dotenv

import re


In [2]:
load_dotenv()
project_root = Path(os.environ["PROJECT_ROOT"])

UNIPROT_PARQUET = project_root / "data/raw/uniprot_human_reviewed.parquet"

In [3]:
df = pd.read_parquet(UNIPROT_PARQUET)
df.shape


(20431, 18)

In [4]:
df.head(1)


,Entry,Entry Name,Gene Names,Protein names,Length,Function [CC],Involvement in disease,Subcellular location [CC],Interacts with,Domain [FT],Region,Zinc finger,Active site,Binding site,Disulfide bond,Modified residue,Natural variant,Tissue specificity
0,A0A0C5B5G6,MOTSC_HUMAN,MT-RNR1,Mitochondrial-derived peptide MOTS-c (Mitochon...,16,FUNCTION: Regulates insulin sensitivity and me...,None,SUBCELLULAR LOCATION: Secreted {ECO:0000269|Pu...,Q16236,None,None,None,None,None,None,None,"VARIANT 14; /note=""K -> Q (specific to the Nor...",TISSUE SPECIFICITY: Detected in plasma (at pro...


In [5]:
df[df["Gene Names"].str.contains("BRCA", na=False)].values[1]
# df[df["Gene Names"].str.contains("ERVK-19", na=False)]
# df[(df['Entry'] == 'P08865') | (df['Entry'] == 'A0A8I5KQE6')]


array(['P38398', 'BRCA1_HUMAN', 'BRCA1 RNF53',
       'Breast cancer type 1 susceptibility protein (EC 2.3.2.27) (RING finger protein 53) (RING-type E3 ubiquitin transferase BRCA1)',
       1863,
       "FUNCTION: E3 ubiquitin-protein ligase that specifically mediates the formation of 'Lys-6'-linked polyubiquitin chains and plays a central role in DNA repair by facilitating cellular responses to DNA damage (PubMed:10500182, PubMed:12887909, PubMed:12890688, PubMed:14976165, PubMed:16818604, PubMed:17525340, PubMed:19261748). It is unclear whether it also mediates the formation of other types of polyubiquitin chains (PubMed:12890688). The BRCA1-BARD1 heterodimer coordinates a diverse range of cellular pathways such as DNA damage repair, ubiquitination and transcriptional regulation to maintain genomic stability (PubMed:12890688, PubMed:14976165, PubMed:20351172). Regulates centrosomal microtubule nucleation (PubMed:18056443). Required for appropriate cell cycle arrests after ionizing ir

In [6]:
df['Function [CC]'].head(20).to_list()
# df['Involvement in disease'].head(20).to_list()
# df['Tissue specificity'].head(20).to_list()


["FUNCTION: Regulates insulin sensitivity and metabolic homeostasis (PubMed:25738459, PubMed:33468709). Inhibits the folate cycle, thereby reducing de novo purine biosynthesis which leads to the accumulation of the de novo purine synthesis intermediate 5-aminoimidazole-4-carboxamide (AICAR) and the activation of the metabolic regulator 5'-AMP-activated protein kinase (AMPK) (PubMed:25738459). Protects against age-dependent and diet-induced insulin resistance as well as diet-induced obesity (PubMed:25738459). In response to metabolic stress, translocates to the nucleus where it binds to antioxidant response elements (ARE) present in the promoter regions of a number of genes and plays a role in regulating nuclear gene expression in an NFE2L2-dependent manner and increasing cellular resistance to metabolic stress (PubMed:29983246). Increases mitochondrial respiration and levels of CPT1A and cytokines IL1B, IL6, IL8, IL10 and TNF in senescent cells (PubMed:29886458). Increases activity of 

In [7]:
df_clean = pd.read_parquet(project_root / "data/processed/uniprot_clean.parquet")
print("proteins:", len(df_clean), "| genes:", df_clean["Gene Names"].str.split().str[0].nunique())

proteins: 5330 | genes: 5325


In [8]:

cov = pd.Series({f: df_clean[f].notna().mean() for f in df_clean.columns})
(cov.sort_values(ascending=False) * 100).round(0).astype(int).rename("non_null_%")

Entry                        100
Entry Name                   100
Gene Names                   100
Protein names                100
Length                       100
Involvement in disease       100
Function [CC]                 98
Subcellular location [CC]     95
Natural variant               94
Interacts with                77
Region                        69
Tissue specificity            65
Modified residue              63
Domain [FT]                   46
Binding site                  35
Disulfide bond                20
Active site                   16
Zinc finger                    8
Name: non_null_%, dtype: int64

*Domain* missing 54%, means no formally annotated domain. Potential reasons:

1. Short proteins / peptides — too small to contain 
   a formally recognised domain (< ~50 aa)
   
2. Intrinsically disordered proteins — no stable 
   3D domain structure to annotate
   
3. Novel/poorly studied proteins — domain exists
   but hasn't been characterised yet in UniProt
   
4. Proteins where function is known but no Pfam/InterPro
   entry has been created for their specific fold

Only enzyme-type proteins have *active sites*. Non-enzymes (structural proteins, scaffolds, adaptor proteins) legitimately have none. Low coverage is expected and correct, not a data quality problem.

Only a minority of proteins contain *zinc fingers*. 8% is actually reasonable given what zinc fingers are.

*Modified residue* — High for signalling and regulatory proteins, low for metabolic enzymes.

*Disulfide bond* — useful when present

*Binding site* — only proteins with experimentally characterised ligand binding get this annotation. Absence doesn't mean no binding occurs, just that it hasn't been formally documented.

*Interacts with* — the 23% null are likely under-studied proteins (less evidence overall).

Tissue specificity — well-studied proteins have tissue annotations, novel ones don't.